# BorakBot — Step 5: the graded evaluation

Fine-tuned against un-fine-tuned, same base, same tokenizer, same 63-item test
split. This is the comparison the rubric asks for; the smoke test in
`colab_finetune.ipynb` Cell 6 was 20 probes and is not a substitute.

**Runtime → Change runtime type → T4 GPU.** About 35 minutes end to end.

Deliberately separate from the training notebook. Nothing here needs
LLaMA-Factory, Drive, or `stage2_chatbot/training/data/` — the adapter comes off the Hub, so
this runs from a clean session in a fraction of the setup time.

    base model  -> Hugging Face   (~6 GB, gated)
    adapter     -> YongVay/borakbot-qlora-r1  (private, 24 MB)
    output      -> stage2_chatbot/eval/results/{base,tuned}.json, score_report.csv
                   -> copied to Drive at the end, because /content is wiped

Produces four numbers per model — perplexity, BLEU, ROUGE-L, BERTScore — plus
chrF, and the refusal rates on the full 11 out-of-scope items rather than 3.

## Cell 1 — Setup

Clone and install. Two to three minutes.

`peft` is needed to attach the adapter, `bitsandbytes` for the 4-bit load. The
three metric packages are eval-only and deliberately **not** in
`requirements.txt`, which describes the CPU environment that serves the
Streamlit app — `bert-score` alone would drag a second copy of the metric stack
onto the demo laptop for no reason.

Load in 4-bit here for the same reason training did: it is what the T4 fits, and
it is what the demo will run. Scoring an fp16 model would measure a
configuration nobody will ever serve.

In [ ]:
import os, pathlib, shutil, subprocess

REPO = pathlib.Path('/content/NLP-BorakBot')
URL  = 'https://github.com/yongvay/NLP_BorakBot.git'

os.chdir('/content')

!pip install -q transformers accelerate peft bitsandbytes
!pip install -q sacrebleu rouge-score bert-score

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '-q', URL, str(REPO)], check=True, cwd='/content')

%cd /content/NLP-BorakBot
!git log --oneline -1

# stage2_chatbot/eval/score.py is the deliverable this notebook exists to run. If the clone
# predates it, everything below fails four cells later instead of here.
assert (REPO / 'stage2_chatbot' / 'eval' / 'score.py').exists(), (
    'STALE CLONE: stage2_chatbot/eval/score.py is not in the pushed repo. Commit and push it, '
    'then re-run this cell.')
print('clone ok')

import torch
print('cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Cell 2 — Hugging Face token

**Required twice over.** The base model is gated, and the adapter repo is
private — so a read-only token is enough here, but it must be the account that
accepted Meta's licence and owns the adapter.

Key icon in the left sidebar: name `HF_TOKEN`, *Notebook access* on. Never paste
the token into a cell.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(userdata.get('HF_TOKEN'))
print('HF login ok')

## Cell 3 — Generate: base, then tuned

Two passes over the same 63 questions, ~12 minutes each. `--all` is what makes
this the full split rather than the committed 20-item probe set — without it
`generate.py` silently reuses `probe_set.jsonl` and you get the smoke test again.

`--tag base` deliberately re-runs the un-fine-tuned model rather than reusing
`stage2_chatbot/eval/results/llama.json` from the bake-off. That file covers 20 probes, not 63,
and the scorer refuses to compare runs built over different question sets.

Greedy decoding both times, so the only variable is the adapter.

In [ ]:
BASE    = 'meta-llama/Llama-3.2-3B-Instruct'
ADAPTER = 'YongVay/borakbot-qlora-r1'

!python stage2_chatbot/eval/generate.py --model {BASE} --split test --all --4bit --tag base

In [ ]:
!python stage2_chatbot/eval/generate.py --model {BASE} --adapter {ADAPTER} \
    --split test --all --4bit --tag tuned

## Cell 4 — Score

Perplexity needs the weights, so this reloads both models; the other three
metrics run off the JSON. Roughly 10 minutes, most of it perplexity.

Read the table with decision 1 from `stage2_chatbot/eval/score.py` in mind: **the 11 refusal
rows are excluded from BLEU/ROUGE/BERTScore.** Their golds are all the same
sentence, and the un-fine-tuned base declines most questions — leaving them in
would pay the base for its worst failure and flatter it by roughly 17% of the
corpus. Refusals are scored separately, and properly, in Cell 5.

`--by-stratum` is where the factual-consistency argument lives. A high average
hiding a collapse in one domain is the thing to look for, and `slang_meanings`
against `jpj_vehicle_licence` is where the smoke test already showed a spread.

In [ ]:
!python stage2_chatbot/eval/score.py --runs base,tuned --by-stratum --perplexity \
    --model {BASE} --adapter-for tuned={ADAPTER} --4bit

## Cell 5 — Refusal rates on the full test split

11 out-of-scope items rather than the smoke test's 3, which is the sample size
Part B needs before it can claim anything about hallucination control.

All five tags in one table so the mallam bake-off row survives — regenerating
the CSV from a subset would drop the evidence behind the base-model choice.
`llama`/`mallam`/`tuned_smoke` are the 20-probe bake-off; `base`/`tuned` are the
63-item split. Different denominators, same file: read the `n_refusal` column
before comparing rows.

In [ ]:
!python stage2_chatbot/eval/refusal_report.py --runs llama,mallam,tuned_smoke,base,tuned --show-misses

## Cell 6 — Get the results out before the session dies

`/content` is wiped on disconnect. These five files are the graded evidence and
cost 35 minutes of GPU to produce.

Copy to Drive, then commit them from the laptop — they are small JSON and CSV,
and `stage2_chatbot/eval/results/` is committed on purpose.

In [ ]:
import pathlib, shutil

OUT = pathlib.Path('/content/drive/MyDrive/RDS3S1/NLP/round1')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT.mkdir(parents=True, exist_ok=True)
except Exception as exc:
    print('Drive unavailable, download by hand from the file browser:', exc)
    OUT = None

WANTED = ['base.json', 'tuned.json', 'score_report.csv', 'refusal_report.csv']
for name in WANTED:
    src = pathlib.Path('stage2_chatbot/eval/results') / name
    if not src.exists():
        print('MISSING', name, '-- re-run the cell that produces it')
        continue
    if OUT:
        shutil.copy2(src, OUT / name)
    print(f'{name:22} {src.stat().st_size:>9,} bytes')

if OUT:
    print('\nsaved to', OUT)

## Cell 7 — What is still outstanding

1. **Commit** `stage2_chatbot/eval/results/{base,tuned}.json`, `score_report.csv` and the
   regenerated `refusal_report.csv`.
2. **Record the numbers in the Part B report**, next to the loss curve — the
   section is written and has the training half already.
3. **The human Likert pass**, which is separate and still required. The report
   §8 explains why the *model-choice* ratings were dropped and states plainly
   that this one is not discharged by that argument. The blinded sheet already
   exists: `stage2_chatbot/eval/make_rating_sheet.py`, scored by `stage2_chatbot/eval/score_ratings.py`.
4. **The prompt ablation**, if time allows — it separates the two hallucination
   safeguards by showing how much refusal behaviour survives without the system
   prompt:

       python stage2_chatbot/eval/generate.py --model <base> --adapter <adapter> --split test \
           --all --no-system-prompt --refusals-only --tag tuned_noprompt
       python stage2_chatbot/eval/refusal_report.py --runs tuned,tuned_noprompt

Then Stage 2-4: `app/normalise.py`, `app/inference.py`, `app/feedback.py`.